In [ ]:
print("dj")

dj


## Sparse Vector Retrival

TF-IDF - Evaluate How important a word is to a document within a collection. 
         Higher weights to the woard that are unique to adocument ( a, are , is the... not give the weights)

BM25 Retrieval:- Keyword base retrieval, document ranked by BM score and bm25 score based on teams         frequency

## Deal Encoder Model
Application:- FAQ system, Document search, Engines, Knowledge base questioning answering

In [ ]:
from sentence_transformers import SentenceTransformer
from pgvector.psycopg2 import register_vector
import psycopg2

# ----------------------------
# Load embedding model
# ----------------------------
model = SentenceTransformer("BAAI/bge-m3")

query = "Transfer of Responsibility"

query_embedding = model.encode(
    query,
    normalize_embeddings=True
).tolist()

# ----------------------------
# Connect to PostgreSQL
# ----------------------------
conn = psycopg2.connect(
    host="localhost",
    port=5434,
    user="admin",
    password="pass123",
    dbname="RAG_POC"
)

register_vector(conn)

cur = conn.cursor()

# ----------------------------
# Semantic Search
# ----------------------------
search_query = """
SELECT
    page_content,
    source,
    metadata
FROM rag_chunks   
ORDER BY embedding <=> %s::vector
LIMIT 5;
"""

# Convert Python list to Postgres array string
embedding_str = "[" + ",".join(str(x) for x in query_embedding) + "]"

cur.execute(search_query, (embedding_str,))

rows = cur.fetchall()

for i, row in enumerate(rows, start=1):
    print(f"\nResult {i}")
    print("=" * 80)
    print("Source :", row[1])
    print("Metadata:", row[2])
    print("Content :")
    print(row[0])

cur.close()
conn.close()

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 45046.77it/s]



Result 1
Source : ABB 800xA.pdf
Metadata: {'source': 'ABB 800xA.pdf', 'chunk_id': '3cbcfbc9-0fe8-4ddd-beeb-d0b4d5420299'}
Content :
The Point of Control functionality allows responsibility interaction from any object that belongs to a section based on the following three protocols:
- Request Responsibility.
- Grab Responsibility.
- Release Responsibility.

The Point of Control functionality allows responsibility interaction from any object that belongs to a section based on the following three protocols:
- Request Responsibility.
- Grab Responsibility.
- Release Responsibility.

Transfer of Responsibility

Headings: Transfer of Responsibility

Result 2
Source : ABB 800xA.pdf
Metadata: {'source': 'ABB 800xA.pdf', 'chunk_id': '4a5ab2f1-4605-4ecf-8486-fb42e5c2514b'}
Content :
The responsibility of a section can be requested using the object context menu. When a user requests the responsibility of a section, a tree structure of the section (with all the subsections) is displayed. The user

In [25]:
import psycopg2
from pgvector.psycopg2 import register_vector
from sentence_transformers import SentenceTransformer
import requests
import json

# ----------------------------
# Config
# ----------------------------
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "qwen2.5:7b"

# ----------------------------
# Embedding model
# ----------------------------
embed_model = SentenceTransformer("BAAI/bge-m3")

# ----------------------------
# Postgres connection
# ----------------------------
conn = psycopg2.connect(
    host="localhost",
    port=5434,
    user="admin",
    password="pass123",
    dbname="RAG_POC"
)
register_vector(conn)
cur = conn.cursor()

# ----------------------------
# Function: retrieve from pgvector
# ----------------------------
def retrieve_chunks(query, k=5):
    query_embedding = embed_model.encode(query, normalize_embeddings=True).tolist()
    embedding_str = "[" + ",".join(str(x) for x in query_embedding) + "]"

    search_sql = """
    SELECT page_content, source, metadata
    FROM rag_chunks
    ORDER BY embedding <=> %s::vector
    LIMIT %s;
    """
    cur.execute(search_sql, (embedding_str, k))
    return cur.fetchall()

# ----------------------------
# Function: ask Ollama with context (collects full answer)
# ----------------------------
def ask_ollama(question, context_chunks):
    context_text = "\n\n".join([chunk[0] for chunk in context_chunks])

    prompt = f"""
Use the following context to answer the question.
If the answer is not available in the context, say "Data is not available."

Context:
{context_text}

Question:
{question}

Answer:
"""

    response = requests.post(
        OLLAMA_URL,
        json={"model": OLLAMA_MODEL, "prompt": prompt, "options": {"temperature": 0}},
        stream=True
    )

    final_answer = []
    for line in response.iter_lines():
        if line:
            data = json.loads(line.decode("utf-8"))
            if "response" in data:
                final_answer.append(data["response"])
            if data.get("done", False):
                break

    return "".join(final_answer).strip()

# ----------------------------
# Example usage
# ----------------------------
if __name__ == "__main__":
    user_query = "How many types of Transfer of Responsibility? and give me only points not extra discription"

    # Step 1: Retrieve chunks
    chunks = retrieve_chunks(user_query, k=5)

    # Step 2: Ask Ollama with retrieved context
    answer = ask_ollama(user_query, chunks)

    print("\nFinal Answer:\n", answer)
    print("\nSource documents:")
    for i, row in enumerate(chunks, start=1):
        print(f"\nResult {i}")
        print("=" * 80)
        print("Source :", row[1])
        print("Metadata:", row[2])
        print("Content :", row[0][:300], "...\n")

    cur.close()
    conn.close()

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 28930.32it/s]



Final Answer:
 

Source documents:

Result 1
Source : ABB 800xA.pdf
Metadata: {'source': 'ABB 800xA.pdf', 'chunk_id': '3cbcfbc9-0fe8-4ddd-beeb-d0b4d5420299'}
Content : The Point of Control functionality allows responsibility interaction from any object that belongs to a section based on the following three protocols:
- Request Responsibility.
- Grab Responsibility.
- Release Responsibility.

The Point of Control functionality allows responsibility interaction from ...


Result 2
Source : ABB 800xA.pdf
Metadata: {'source': 'ABB 800xA.pdf', 'chunk_id': 'af594d38-7094-456d-ad66-496278f8ed01'}
Content : Approval (Authentication), = ..............................................................................................37. Point of Control...............................................................................................................39, = . Transfer of Responsibility............ ...


Result 3
Source : ABB 800xA.pdf
Metadata: {'source': 'ABB 800xA.pdf', 'chunk_id': '4a

In [11]:
import psycopg2
from pgvector.psycopg2 import register_vector

conn = psycopg2.connect(
    host='localhost',
    port=5434,
    user='admin',
    password='pass123',
    dbname='RAG_POC'
)

register_vector(conn)

cur = conn.cursor()

chunk_id = "cf5cd03c-cab3-4dd0-85cf-ac229061188b"

cur.execute(
    """
    SELECT page_content
    FROM rag_chunks
    WHERE chunk_id = %s;
    """,
    (chunk_id,)
)

row = cur.fetchone()

if row:
    text = row[0]
    print("Text Length:", len(text))
    print(text)
else:
    print("No record found.")

cur.close()
conn.close()

Text Length: 8098
Section 11 - Structured Data Logger, = . SDL Data View..............................................................................................................239, = . Appendix A - System Alarm Messages, = . Operations......................................................................................................................242, = . Device Management Foundation Fieldbus ....................................................................245, = . Batch Management........................................................................................................246, = . 800xA History, = ...............................................................................................................247. PC, Network Software and Monitoring (PNSM), = ..........................................................248. 800xA for Advant Master..............................................................................................250, = . Melody..............

In [ ]:
from sentence_transformers import SentenceTransformer
from pgvector.psycopg2 import register_vector
import psycopg2
import ollama

# ==========================================================
# Configuration
# ==========================================================

HOST = "localhost"
PORT = 5434
USER = "admin"
PASSWORD = "pass123"
DATABASE = "RAG_POC"

TABLE = "rag_chunks"

EMBED_MODEL = "BAAI/bge-m3"
OLLAMA_MODEL = "qwen3:8b"

TOP_K = 5

# Query used for semantic search
SEARCH_QUERY = "if my application bar is configure so in which figure number i can see in Alarm Logger Manager"

# User question
QUESTION = "if my application bar is configure so in which figure number i can see in Alarm Logger Manager"

# ==========================================================
# Load Embedding Model
# ==========================================================

print("Loading embedding model...")

embed_model = SentenceTransformer(EMBED_MODEL)

query_embedding = embed_model.encode(
    SEARCH_QUERY,
    normalize_embeddings=True
).tolist()

embedding_str = "[" + ",".join(map(str, query_embedding)) + "]"

# ==========================================================
# PostgreSQL Connection
# ==========================================================

print("Connecting to PostgreSQL...")

conn = psycopg2.connect(
    host=HOST,
    port=PORT,
    user=USER,
    password=PASSWORD,
    dbname=DATABASE
)

register_vector(conn)

cur = conn.cursor()

# ==========================================================
# Semantic Search
# ==========================================================

sql = f"""
SELECT
    page_content,
    source,
    metadata
FROM {TABLE}
ORDER BY embedding <=> %s::vector
LIMIT %s;
"""

cur.execute(sql, (embedding_str, TOP_K))

rows = cur.fetchall()

cur.close()
conn.close()

if len(rows) == 0:
    print("No documents found.")
    exit()

# ==========================================================
# Build Context
# ==========================================================

context = ""

# print("\nRetrieved Documents")
# print("=" * 80)

for i, (page_content, source, metadata) in enumerate(rows, start=1):

    # print(f"\nChunk {i}")
    # print("-" * 80)
    # print("Source :", source)
    # print("Metadata :", metadata)
    # print(page_content)

    context += page_content + "\n\n"

# ==========================================================
# Prompt
# ==========================================================

prompt = f"""
You are a RAG assistant.

Answer ONLY from the supplied context.

If the answer is not present in the context, reply exactly:

Data is not available.

Context:
{context}

Question:
{QUESTION}

Rules:
1. Use ONLY the context.
2. Do NOT use outside knowledge.
3. Return only the requested points.
4. Do not explain.
5. Do not summarize.
6. Keep the wording from the context.

Answer:
"""

# ==========================================================
# Ask Ollama
# ==========================================================

print("\nGenerating answer using Ollama...\n")

response = ollama.chat(
    model=OLLAMA_MODEL,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print("=" * 80)
print("FINAL ANSWER")
print("=" * 80)

print(response["message"]["content"])

c:\RAG_POC\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embedding model...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 33607.38it/s]


Connecting to PostgreSQL...

Generating answer using Ollama...

FINAL ANSWER
| Acknowledge Page, Hot Key = CTRL+P |  
| Acknowledge Whole List, Hot Key = CTRL+L |  
| Print Alarm List, Hot Key = CTRL+P |  
| Print Selected Alarms, Hot Key = CTRL+P |  
| Print All Alarms, Hot Key = CTRL+P |  
| Print Alarm List with Details, Hot Key = CTRL+P |  
| Print Alarm List with Timestamps, Hot Key = CTRL+P |  
| Print Alarm List with Status, Hot Key = CTRL+P |  
| Print Alarm List with Source, Hot Key = CTRL+P |  
| Print Alarm List with Priority, Hot Key = CTRL+P |  
| Print Alarm List with Category, Hot Key = CTRL+P |  
| Print Alarm List with Description, Hot Key = CTRL+P |  
| Print Alarm List with Additional Info, Hot Key = CTRL+P |
